In [ ]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt # we only need pyplot
sb.set()

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder


from plotly.subplots import make_subplots
import plotly.offline as py
import plotly.graph_objects as go


from matplotlib.dates import YearLocator, MonthLocator, DateFormatter, WeekdayLocator, DayLocator
%matplotlib inline
import time
import re
from datetime import datetime
import matplotlib.dates as mdates
from wordcloud import WordCloud, STOPWORDS

from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sklearn.tree import export_graphviz
from sklearn.metrics import confusion_matrix

from scipy.ndimage.filters import gaussian_filter1d

import nltk

from nltk.corpus import stopwords
import string



# Correlation between host information and price


In [ ]:
calendar_data = pd.read_csv('calendar.csv')
calendar_data.head()

In [ ]:
listing_data = pd.read_csv('listings.csv')
listing_data.head()

In [ ]:
listing_data.info()

In [ ]:
calendar_data.shape

In [ ]:
listings = pd.DataFrame(listing_data[['host_location','host_response_time','host_response_rate','host_acceptance_rate','host_is_superhost','host_verifications','neighbourhood_cleansed','latitude','longitude','property_type','room_type','accommodates','bathrooms','bedrooms','beds','bed_type','amenities','cleaning_fee','guests_included','minimum_nights','number_of_reviews','review_scores_accuracy','review_scores_cleanliness','review_scores_checkin','review_scores_communication','review_scores_location','review_scores_value','cancellation_policy','reviews_per_month','price']])
from sklearn.preprocessing import LabelEncoder

listings['price'] = listings['price'].replace('[\$,]', '', regex=True).astype(float)
listings['price'] = listings['price'].fillna(0)
listings['bathrooms'] = listings['bathrooms'].fillna(0)
listings['bedrooms'] = listings['bedrooms'].fillna(0)
listings['beds'] = listings['beds'].fillna(0)
listings['review_scores_accuracy'] = listings['review_scores_accuracy'].fillna(0)
listings['review_scores_cleanliness'] = listings['review_scores_cleanliness'].fillna(0)
listings['review_scores_checkin'] = listings['review_scores_checkin'].fillna(0)
listings['review_scores_communication'] = listings['review_scores_communication'].fillna(0)
listings['review_scores_location'] = listings['review_scores_location'].fillna(0)
listings['review_scores_value'] = listings['review_scores_value'].fillna(0)
listings['reviews_per_month'] = listings['reviews_per_month'].fillna(0)

number = LabelEncoder()
listings['host_is_superhost']=number.fit_transform(listings['host_is_superhost'].astype('str'))
listings['property_type']=number.fit_transform(listings['property_type'].astype('str'))
listings['room_type']=number.fit_transform(listings['room_type'].astype('str'))
listings['bed_type']=number.fit_transform(listings['bed_type'].astype('str'))
listings['cleaning_fee']=number.fit_transform(listings['cleaning_fee'].astype('str'))
listings['host_location']=number.fit_transform(listings['host_location'].astype('str'))
listings['host_response_time']=number.fit_transform(listings['host_response_time'].astype('str'))
listings['host_response_rate']=number.fit_transform(listings['host_response_rate'].astype('str'))
listings['host_acceptance_rate']=number.fit_transform(listings['host_acceptance_rate'].astype('str'))
listings['host_verifications']=number.fit_transform(listings['host_verifications'].astype('str'))
listings['neighbourhood_cleansed']=number.fit_transform(listings['neighbourhood_cleansed'].astype('str'))
listings['amenities']=number.fit_transform(listings['amenities'].astype('str'))
listings['cancellation_policy']=number.fit_transform(listings['cancellation_policy'].astype('str'))

listings.info()

In [ ]:
X = listings.drop(columns = 'price')
y = listings[['price']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=40)

randomfor = RandomForestRegressor(n_estimators=100,
                               criterion='squared_error',
                               random_state=40,
                               n_jobs=-1)
randomfor.fit(X_train, y_train.squeeze())

#calculate scores for the model
y_train_preds = randomfor.predict(X_train)
y_test_preds = randomfor.predict(X_test)

print('Random Forest MSE train:, test:', (mean_squared_error(y_train, y_train_preds), mean_squared_error(y_test, y_test_preds)))
print('Random Forest R^2 train:, test:', (r2_score(y_train, y_train_preds), r2_score(y_test, y_test_preds)))

headers = ["name", "score"]
values = sorted(zip(X_train.columns, randomfor.feature_importances_), key=lambda x: x[1] * -1)
feature_impt = pd.DataFrame(values, columns = headers)
feature_impt = feature_impt.sort_values(by = ['score'], ascending = False)

features = feature_impt['name'][:29]
y_pos = np.arange(len(features))
scores = feature_impt['score'][:29]

plt.figure(figsize=(20,10))
plt.bar(y_pos, scores, align='center', alpha=0.5)
plt.xticks(y_pos, features, rotation='vertical')
plt.ylabel('Score')
plt.xlabel('Features')
plt.title('Feature importance (Random Forest)')


plt.show()

From the above random forest regression we can choose which features to focus on.

Focus on host verification, cleaning fee, reviews per month, host location, host response time, host response rate, host acceptance rate

In [ ]:
newlisting = pd.DataFrame(listing_data[['host_verifications', 'cleaning_fee', 'reviews_per_month', 'host_location', 'host_response_rate', 'host_response_time', 'host_acceptance_rate', 'price']])
newlisting.head()

In [ ]:
#Remove $ sign
newlisting['price'] = newlisting['price'].apply(lambda x: str(x).replace('$',''))
newlisting['price'] = pd.to_numeric(newlisting['price'], errors = 'coerce')
newlisting['price']

In [ ]:
#Remove $ sign
newlisting['cleaning_fee'] = newlisting['cleaning_fee'].apply(lambda x: str(x).replace('$',''))
newlisting['cleaning_fee'] = pd.to_numeric(newlisting['cleaning_fee'], errors = 'coerce')
newlisting['cleaning_fee']

In [ ]:
#Remove % sign
newlisting['host_response_rate'] = newlisting['host_response_rate'].apply(lambda x: str(x).replace('%',''))
newlisting['host_response_rate'] = pd.to_numeric(newlisting['host_response_rate'], errors = 'coerce')
newlisting['host_response_rate']

In [ ]:
#Remove % sign
newlisting['host_acceptance_rate'] = newlisting['host_acceptance_rate'].apply(lambda x: str(x).replace('%',''))
newlisting['host_acceptance_rate'] = pd.to_numeric(newlisting['host_acceptance_rate'], errors = 'coerce')
newlisting['host_acceptance_rate']

In [ ]:
#change the host_response_time to categorical numbers.
def timeconversion(t):
    if t == 'within an hour':
        t = '1'
    elif t == 'within a few hours':
        t = '2'
    elif t == 'within a day':
        t = '3'
    elif t == 'a few days or more':
        t = '4'
    else:
        t = '0'
    return t


newlisting['host_response_time'] = newlisting['host_response_time'].apply(timeconversion).astype(int)
newlisting['host_response_time']

In [ ]:
#Convert the host_verifications to numeric variables
def vericonversion(a):
    value = a.replace('[','').replace(']','').replace(' ','').replace(',',' ')
    value = value.split()
    return value

newlisting['array_host_verifications'] = newlisting['host_verifications'].apply(lambda a: vericonversion(a))
newlisting['host_verifications'] = newlisting['host_verifications'].apply(lambda a: len(vericonversion(a)))
newlisting['host_verifications']

In [ ]:
newlisting = newlisting.fillna(0)
newlisting.head()

In [ ]:
numlisting = newlisting.drop(columns = ['host_location', 'array_host_verifications'])
numlisting.head()

In [ ]:
numlisting.describe()

In [ ]:
# Draw the distributions of all variables
f, axes = plt.subplots(7, 3, figsize=(18, 24))

count = 0
for var in numlisting:
    sb.boxplot(data = numlisting[var], orient = "h", ax = axes[count,0])
    sb.histplot(data = numlisting[var], ax = axes[count,1])
    sb.violinplot(data = numlisting[var], orient = "h", ax = axes[count,2])
    count += 1

In [ ]:
sb.pairplot(data = numlisting)

In [ ]:
#Comparing the correlation between different variables
f, axes = plt.subplots(1, 1, figsize=(10, 10))
sb.heatmap(numlisting.corr(), vmin = -1, vmax = 1, linewidths = 1,
           annot = True, fmt = ".2f", annot_kws = {"size": 18}, cmap = "RdBu")

In [ ]:
hostloca = pd.DataFrame(newlisting['host_location'])
hostloca.describe()

In [ ]:
hostloca = newlisting['host_location'].str.contains('Seattle', na = False)
hostloca

In [ ]:
sb.countplot(hostloca)

In [ ]:
price = pd.DataFrame(newlisting['price'])
price.head()

In [ ]:
joinDF = pd.concat([price, hostloca], axis = 1).reindex(price.index)

f = plt.figure(figsize=(30, 12))
sb.swarmplot(x = "price", y = "host_location", data = joinDF, orient = "h")

In [ ]:
dectree = DecisionTreeClassifier(max_depth = 3)

In [ ]:
dectree.fit(price, hostloca)

In [ ]:
f = plt.figure(figsize=(12,12))
plot_tree(dectree, filled=True, rounded=True,
          feature_names=["price"]
          )

In [ ]:
hostloca_pred = dectree.predict(price)

# Print the Classification Accuracy
print("Classification Accuracy \t:", dectree.score(price, hostloca))

# Plot the two-way Confusion Matrix
sb.heatmap(confusion_matrix(hostloca, hostloca_pred),
           annot = True, fmt=".0f", annot_kws={"size": 18})

In [ ]:
superhost = listing_data['host_is_superhost'].str.contains('t', na = False)
superhost

In [ ]:
sb.countplot(superhost)

In [ ]:
combDF = pd.concat([price, superhost], axis = 1).reindex(price.index)

f = plt.figure(figsize=(30, 12))
sb.swarmplot(x = "price", y = "host_is_superhost", data = combDF, orient = "h")

In [ ]:
dectree.fit(price, superhost)

In [ ]:
f = plt.figure(figsize=(12,12))
plot_tree(dectree, filled=True, rounded=True,
          feature_names=["price"]
          )

In [ ]:
superhost_pred = dectree.predict(price)

# Print the Classification Accuracy
print("Classification Accuracy \t:", dectree.score(price, superhost))

# Plot the two-way Confusion Matrix
sb.heatmap(confusion_matrix(superhost, superhost_pred),
           annot = True, fmt=".0f", annot_kws={"size": 18})

# Calendar Date and Location vs Price and predicting visitors for 2017

In [ ]:
calendar = pd.read_csv("calendar.csv")

In [ ]:
# cleaning numerical data
# convert price column to numeric by removing "$"
# changing NaN to 0

calendar['price'] = calendar['price'].apply(lambda x: str(x).replace('$',''))
calendar['price'] = pd.to_numeric(calendar['price'], errors = 'coerce')

calendar = calendar.fillna(0)
calendar.head(10)

In [ ]:
calendar.info()

### 1) Calendar Date vs Price

In [ ]:
# Step 1) Organise data by dates
# organising the data by sorting into the different dates and finding the total sum of price, average price, standard deviation based on individual dates.
# grouping the date category and applying price functions to the category.

calendardata           = calendar.groupby('date')[["price"]].sum()
calendardata['Avg']    = calendar.groupby('date')[["price"]].mean()
calendardata['Std']    = calendar.groupby('date')[["price"]].std()
calendardata.columns   = ['Total', 'Avg', 'Std']

calendardata.head(10)

In [ ]:
# Step 2) Include year, months, days, weekdays, week numbers
# Split date column into year, month, day, weekday columns
# Start weekday with Monday=0, Sunday=6
# Set the range of weekends from Friday to Sunday

calendar['year'] = pd.DatetimeIndex(calendar['date']).year
calendar['month'] = pd.DatetimeIndex(calendar['date']).month
calendar['day'] = pd.DatetimeIndex(calendar['date']).day
calendar['weekday'] = pd.DatetimeIndex(calendar['date']).weekday
calendar['weekno'] = pd.DatetimeIndex(calendar['date']).week
calendar.head(20)

In [ ]:
# Step 3) plot graph of calendar date vs average price
# Step 4) Plot graph of calendar date vs standard deviation

calendardata['Avg'].plot(figsize=(15, 8)) # blue line
calendardata['Std'].plot(figsize=(15, 8)) # orange line

In [ ]:
sb.jointplot(data = calendardata, x = "date" , y = "Avg", height = 12)

In [ ]:
calendardata.corr()

In [ ]:
# Step 5) Using scatter plot to compare calendar date and average price
# Shows individual dates average prices

fig = make_subplots(rows = 1, cols = 1, subplot_titles = ("Calendar Date Vs Price"))

p1 = go.Scatter(
        x = calendar['date'],
        y = calendardata['Avg'],
        mode = 'markers', showlegend = False)
fig.append_trace(p1,1, 1)

fig['layout'].update(height = 1000, width = 1000)
py.iplot(fig)

There is a clear increase in average price from january 2016 to december 2016
most increase in average price is after the january period

### Holiday vs price

In [ ]:
# Find the holiday period in 2016

Holiday = pd.read_html('https://www.timeanddate.com/calendar/?country=1&year=2016')

In [ ]:
print("HTML tables : ", len(Holiday))

In [ ]:
Holiday[13].head(10)

In [ ]:
# Find School Holidays in 2016

SchoolHoliday = pd.read_html('https://www.feiertagskalender.ch/ferien.php?geo=3537&jahr=2016&hl=en')

In [ ]:
print("HTML tables : ", len(SchoolHoliday))

In [ ]:
SchoolHoliday[0].head(10)

In [ ]:
# start date 4th Jan 2016, end date 31st Dec 2016
# count no of days between start date and start of holiday/end of holiday
# show average price range for each holiday

import matplotlib.pyplot as plt

calendardata['Avg'].plot(figsize=(16, 6))

blue   = plt.axvspan(42,  46,  facecolor="blue", alpha=0.3, label = 'winter holiday')
green  = plt.axvspan(81,  95,  facecolor="green", alpha=0.3, label = 'easter holiday')
yellow = plt.axvspan(147, 151, facecolor="yellow", alpha=0.3, label = 'spring holiday')
cyan    = plt.axvspan(168, 240, facecolor='cyan',alpha=0.3, label = 'summer holiday')
red   = plt.axvspan(325, 326, facecolor="red", alpha=0.3, label = 'autumn holiday')
purple = plt.axvspan(353, 362, facecolor="purple", alpha=0.3, label = 'christmas holiday')

plt.show()

### Identifying occupancy rates

In [ ]:
#finding the seasonal pattern

def timeplot(data, feat, title):
    plt.figure(figsize=(20,8))
    x = [datetime.strptime(date, '%Y-%m-%d') for date in data.index]
    y = data[feat]
    # smooth y for visualization
    y_smooth = gaussian_filter1d(y, sigma=5)
    # set x tick by month
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
    plt.title(title)
    plt.plot(x, y_smooth, 'c-')
    plt.show()

In [ ]:
calendar['available'] = calendar['available'].map(lambda available: 1 if available == 't' else 0)

#getting occupancy rate
occupancy = calendar[['date', 'available']].groupby('date').mean()
occupancy['occupancy'] = 1 - occupancy['available']
occupancy.head()

In [ ]:
timeplot(occupancy, 'occupancy', 'Occupancy rate')

In [ ]:
median = calendar[['date', 'price']].groupby('date').median()

timeplot(median, 'price', 'Median Price')

### 2) Longitude / Latitude vs Price

In [ ]:
listings = pd.read_csv("listings.csv")
listings = listings.fillna(0)

longitude = listings['longitude']
latitude = listings['latitude']

In [ ]:
longitude.describe()

In [ ]:
latitude.describe()

In [ ]:
price = listings['price'].replace('[\$,]', '', regex=True).astype(float)
price.head()

In [ ]:
# Scatterplot for longitude vs latitude
# Colorbar to represent the price range

import matplotlib.pyplot as plt

Average = calendardata['Avg']
listings.plot(kind="scatter", x="longitude", y="latitude",label="id", c= price,cmap=plt.get_cmap("nipy_spectral"),
    colorbar = True, alpha=1, figsize=(12,10))
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.image as mpimg

seattle_img = mpimg.imread('seattle.jpg')
ax = listings.plot(kind ="scatter", x = "longitude", y = "latitude", figsize = (20,16),
                   label = "Hosts", c = price, cmap = plt.get_cmap("nipy_spectral"),
                   grid = False, colorbar = False, alpha = 1,)

plt1 = plt.imshow(seattle_img, cmap = 'nipy_spectral', extent = [-122.441, -122.217, 47.505088, 47.733328], alpha = 0.3)
plt.ylabel("Latitude", fontsize = 20)
plt.xlabel("Longitude", fontsize = 20)

tick_values = np.linspace(price.min(), price.max(), 20)
cbar = plt.colorbar(plt1, label = 'Colorbar !')
cbar.ax.set_yticklabels(["$%d" % (round(v)) for v in tick_values], fontsize = 20)
cbar.set_label('price', fontsize = 20)

plt.legend(fontsize=20)
plt.show()

### 3) Estimate the likelihood of booking in year 2017

In [ ]:
# Find the monthly vistors for the year 2016
# Find average visitors for the year 2016

calendar = calendar.dropna()
date = calendar['date']
price = calendar['price']

year_date = {date: 0 for date in set(date)}
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December', 'January']

for i in date:
    year_date[i] += 1
monthly_visitors = np.zeros(12)

for j in year_date:
    monthly_visitors[int(j.split('-')[1]) - 1] += year_date[j]
monthly_visitors = [int(visitors) for visitors in monthly_visitors]

count = 0
for a in monthly_visitors:
    print("Monthly visitors in {:10s} : {:10d}".format(months[count], a))
    count += 1
print()

sum = 0
for b in monthly_visitors:
    sum += int(b)
avg = sum/12

print("Mean monthly visitors in 2016  : ", int(avg))

In [ ]:
from sklearn import linear_model
from sklearn.preprocessing import PolynomialFeatures
from sklearn import svm
from sklearn.model_selection import cross_val_score

y = np.array(monthly_visitors)
y_copy = y
X = np.array(np.arange(1, 13))
X_copy = X

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25)

X_train=np.sort(X_train)
X_test=np.sort(X_test)

X = X_train[:, np.newaxis]
y = y_train[:, np.newaxis]
X_test = X_test[:, np.newaxis]
y_test = y_test[:, np.newaxis]

polyfeatures= PolynomialFeatures(degree=4)
x_poly = polyfeatures.fit_transform(X)
x_poly_test = polyfeatures.fit_transform(X_test)

ridge = linear_model.Ridge()
ridge.fit(x_poly, y)

y_poly_pred = ridge.predict(x_poly)
y_poly_pred_test = ridge.predict(x_poly_test)

print("4 degree Polynomial Ridge Regression")
print("Coefficients and Intercept of the Linear Regression: \n")
print("Y = " + str([round(x,2) for x in ridge.coef_[0]]) + "X + "+ str(round(ridge.intercept_[0], 2)))

from sklearn.metrics import mean_squared_error

print("Goodness of Fit of Model \tTrain Dataset")
print("Explained Variance (R^2) \t:", ridge.score(x_poly, y))
print("Mean Squared Error (MSE) \t:", mean_squared_error(y, y_poly_pred))
print()

print("Goodness of Fit of Model \tTest Dataset")
print("Explained Variance (R^2) \t:", ridge.score(x_poly_test, y_test))
print("Mean Squared Error (MSE) \t:", mean_squared_error(y_test, y_poly_pred_test))
print()

mse_decrease = []
mse_decrease.append((mean_squared_error(y_test, y_poly_pred_test)- mean_squared_error(y, y_poly_pred))/ mean_squared_error(y, y_poly_pred))

plt.figure(figsize=(15,5))
plt.scatter(X, y)
plt.plot(X, y_poly_pred, color="red")
plt.title("Prediction for 2017", fontsize=20)
plt.xlabel("Months", fontsize=16)
plt.ylabel("Exact Visiter Count", fontsize=16)
plt.yticks(np.arange(100000, 130000, 5000), fontsize=16)
plt.xticks(np.arange(1, 14), months, fontsize=12)
plt.show()

r2_list = []
r2_list_test = []

for i in range(0,10):
    y = np.array(monthly_visitors)
    X = np.array(np.arange(1, 13))

    X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.25)
    X_train=np.sort(X_train)
    X_test=np.sort(X_test)

    X = X_train[:, np.newaxis]
    y = y_train[:, np.newaxis]
    X_test = X_test[:, np.newaxis]
    y_test = y_test[:, np.newaxis]

    x_poly = polyfeatures.fit_transform(X)
    x_poly_test = polyfeatures.fit_transform(X_test)
    ridge.fit(x_poly, y)

    y_poly_pred = ridge.predict(x_poly)
    y_poly_pred_test = ridge.predict(x_poly_test)

    r2_list.append(ridge.score(x_poly, y))
    r2_list_test.append(ridge.score(x_poly_test, y_test))

The prediction shows the no of visitors will be roughly the same from january to june and followed by a gradual decrease over the remaining months

# Find the correlation between Property type/information vs Price


In [ ]:
listing_data=pd.read_csv('listings.csv')
listing_data.head()

In [ ]:
new_data = pd.DataFrame(listing_data[['price','property_type','room_type','accommodates','bathrooms','bedrooms','beds','bed_type','amenities','minimum_nights']])
new_data.head()

In [ ]:
#Extract out the numerical data
num_data=pd.DataFrame(new_data[['accommodates','bathrooms','bedrooms','beds','minimum_nights']])
num_data.head()

In [ ]:
price_data=pd.DataFrame(new_data['price'])
price_data['price'] = price_data['price'].apply(lambda x: str(x).replace('$',''))
price_data['price'] = pd.to_numeric(price_data['price'], errors = 'coerce')
price_data

In [ ]:
#Mutual boxplots to identify any existing correlation between the numerical data vs price
f, axes = plt.subplots(5, 1, figsize=(10, 40))

count = 0
for var in num_data:
    if var!='price':
        sb.boxplot(data = num_data, x = price_data['price'], y = var, orient = 'h', ax = axes[count])
        count += 1

In [ ]:
# find the correlation between these variables with price
num_data.corr()

In [ ]:
# Heatmap of the Correlation Matrix
f = plt.figure(figsize=(12, 12))
sb.heatmap(num_data.corr(), vmin = -1, vmax = 1, annot = True, fmt = ".2f")

In [ ]:
sb.pairplot(data = num_data)

In [ ]:
#Extract out the variables that are categorical
cat_data=pd.DataFrame(listing_data[['property_type','room_type','bed_type']])

In [ ]:
# visualise the countplots of the data
for var in cat_data:
    sb.catplot(data = listing_data, y = var, kind = 'count', orient = 'h')
    print(cat_data[var].value_counts())
    print()


In [ ]:
#Mutual boxplots to identify any existing correlation between the categorical data and price
f, axes = plt.subplots(3, 1, figsize=(15, 60))

count = 0
for var in cat_data:
    sb.boxplot(data = cat_data, x = price_data['price'], y = var, orient = 'h', ax = axes[count])
    count += 1

In [ ]:
#Mutual violinplot identify any existing correlation between the categorical data and price
f, axes = plt.subplots(3, 1, figsize=(15, 60))

count = 0
for var in cat_data:
    sb.violinplot(data = cat_data, x = price_data['price'], y = var, orient = 'h', ax = axes[count])
    count += 1

In [ ]:
#Convert the amenities to numeric variables
def amenconversion(a):
    value = a.replace('[','').replace(']','').replace(' ','').replace(',',' ')
    value = value.split()
    return value

new_data['array_amenities'] = new_data['amenities'].apply(lambda a: amenconversion(a))
new_data['amenities'] = new_data['amenities'].apply(lambda a: len(amenconversion(a)))
new_data['amenities']

In [ ]:
amen_data=pd.DataFrame(new_data['amenities'])
amen_data.head()

In [ ]:
# combine all the data together
total_data=pd.concat([num_data,cat_data,amen_data,price_data],axis=1)
total_data.head()

In [ ]:
# we can remove the categories of property_type,room_type and bed_type that have low count since they will not be useful in our analysis
bed_type = ['Couch']
for bedType in bed_type:
    total_data = total_data[total_data['bed_type']!=bedType]

property_type = ['Cabin', 'Camper/RV', 'Bungalow', 'Boat', 'Other', 'Dorm', 'Treehouse', 'Yurt', 'Chalet', 'Tent']
for propertyType in property_type:
    total_data = total_data[total_data['property_type']!=propertyType]

In [ ]:
#Creating a dictionary to assign numeric values to categorical data to allow for further analysis
numcat_dict = {'bed_type':{'Real Bed':0, 'Pull-out Sofa': 1,'Futon':2, 'Airbed':3},
                      'property_type':{'Apartment':0, 'House':1, 'Cabin':2, 'Condominium':3, 'Townhouse':4, 'Loft':5,
                                       'Bed & Breakfast':6},
                      'room_type':{'Entire home/apt':0, 'Private room':1, 'Shared room':2}}

total_data.replace(numcat_dict,inplace= True)
total_data.head()

In [ ]:
#find to correlation of all the data against price
total_data.corr()

In [ ]:
f, axes = plt.subplots(1, 1, figsize=(20,20))
sb.heatmap(total_data.corr(), vmin = -1, vmax = 1, annot = True, fmt = '.3f')

In [ ]:
sb.pairplot(data=total_data)

# ANALYSIS OF USER REVIEWS

In [ ]:
reviews_DF = pd.read_csv("reviews.csv")
reviews_DF = reviews_DF.dropna()
reviews_DF.head()
nltk.download('vader_lexicon')

In [ ]:
# Importing the listing dataset
listings_DF = pd.read_csv('listings.csv')
listings_DF.head()

In [ ]:
# Replace null values with 0
listings_DF.fillna(0, inplace=True)


listings_DF['price'] = listings_DF['price'].apply(lambda x: x[1:])
listings_DF['price'] = listings_DF['price'].apply(lambda x: x.replace(',', ''))
listings_DF['price'] = listings_DF['price'].astype('float')
priceDF = listings_DF['price']

# Remove listings with null values for bathrooms, bedrooms, accomodates, beds, price, reviews_per_month, review_scores_rating
listings_DF = listings_DF[listings_DF.bathrooms > 0]
#listings_DF = listings_DF[listings_DF.price > 0]
listings_DF = listings_DF[listings_DF.bedrooms > 0]
listings_DF = listings_DF[listings_DF.reviews_per_month > 0]
listings_DF = listings_DF[listings_DF.accommodates > 0]
listings_DF = listings_DF[listings_DF.beds > 0]
listings_DF = listings_DF[listings_DF.review_scores_rating > 0]


listings_DF.head()

### Analysing summaries of Most Expensive 150 listings

In [ ]:
# Analysing the dataframe of the summaries of top 150 most expensive listings
summary_DF = listings_DF[['summary','price']]
summary_DF = summary_DF[pd.notnull(summary_DF['summary'])]
summary_DF = summary_DF[summary_DF['summary']!=0]
summary_DF = summary_DF.sort_values('price',ascending=[0])
top150DF = summary_DF.head(100)
top150DF.head()

In [ ]:
# data cleaning for review summaries

words=''
for index,row in top150DF.iterrows():
    words += row['summary']

string_punctuation = string.punctuation
ignoreChar=['\r','\n','',' ',"'s"]
nums=['0','1','2','3','4','5','6','7','8','9']
summary_data=nltk.word_tokenize(words)
words = [l.lower() for l in summary_data if l not in string_punctuation if l not in ignoreChar if l not in nums]
filtered_data=[word for word in words if word not in stopwords.words('english')]
wnl = nltk.WordNetLemmatizer()
final_data=[wnl.lemmatize(data) for data in filtered_data]
finalWords=' '.join(final_data)
finalWords[:50]

In [ ]:


wordcloud = WordCloud(width = 1000, height = 700, background_color="white").generate(finalWords)
plt.figure(figsize=(18,12))
plt.imshow(wordcloud)
plt.axis("off")
plt.show()


### Analysing summaries of Least Expensive 150 listings

In [ ]:
# Creating a dataframe of the summaries of the 100 most cheapest listings
summaryCheapest_DF = listings_DF[['summary','price']]
summaryCheapest_DF = summaryCheapest_DF[pd.notnull(summaryCheapest_DF['summary'])]
summaryCheapest_DF = summaryCheapest_DF[summaryCheapest_DF['summary']!=0]
summaryCheapest_DF = summaryCheapest_DF.sort_values('price',ascending=[1])
cheapest_150DF = summaryCheapest_DF.head(150)
cheapest_150DF.sample(n=5)

In [ ]:
# cleaning data for summaries of lowest listings

words=''
for index,row in cheapest_150DF.iterrows():
    words += row['summary']

string_punctuation = string.punctuation
ignoreChar=['\r','\n','',' ',"'s"]
nums=['0','1','2','3','4','5','6','7','8','9']
summary_data=nltk.word_tokenize(words)
words = [l.lower() for l in summary_data if l not in string_punctuation if l not in ignoreChar if l not in nums]
filtered_data=[word for word in words if word not in stopwords.words('english')]
wnl = nltk.WordNetLemmatizer()
final_data=[wnl.lemmatize(data) for data in filtered_data]
finalWords=' '.join(final_data)
finalWords[:50]

In [ ]:
wordcloud = WordCloud(width = 1000, height = 700, background_color="white").generate(finalWords)
plt.figure(figsize=(18,12))
plt.imshow(wordcloud)
plt.axis("off")
plt.show()

### Correlation between review sentiment and price

In [ ]:
reviews_DF = pd.read_csv("reviews.csv")
reviews_DF = reviews_DF.dropna()
reviews_DF.head()

We are going to use the built-in analyzer in the NLTK Python library to assign a polarity score to each comment to retrieve the sentiment of comments - 'positive','negative' or 'neutral'

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()

reviews_DF['polarity_value']="Default"
reviews_DF['neg']=0.0
reviews_DF['pos']=0.0
reviews_DF['neu']=0.0
reviews_DF['compound']=0.0
for index,row in reviews_DF.iterrows():
    ss = sid.polarity_scores(row['comments'])
    reviews_DF.at[index,'polarity_value'] = ss
    reviews_DF.at[index,'neg'] = ss['neg']
    reviews_DF.at[index,'pos'] = ss['pos']
    reviews_DF.at[index,'neu']= ss['neu']
    reviews_DF.at[index,'compound'] = ss['compound']
reviews_DF.head()


In [ ]:
reviews_DF.to_csv('./polarity_reviews.csv')

In [ ]:
# For better analysis we will be removing comments that are not in English.
from langdetect import detect

def detect_lang(sente):
    sente=str(sente)
    try:
        return detect(sente)
    except:
        return "None"

for index,row in reviews_DF.iterrows():
    lang=detect_lang(row['comments'])
    reviews_DF.at[index,'language'] = lang

Eng_Reviews_DF=reviews_DF[reviews_DF.language=='en']

Eng_Reviews_DF.head()

##### Categorizing into positive, neutral and negative reviews

In [ ]:
polarDF=Eng_Reviews_DF[['pos']]
polarDF=polarDF.groupby(pd.cut(polarDF["pos"], np.arange(0, 1.1, 0.1))).count()
polarDF=polarDF.rename(columns={'pos':'count_of_Comments'})
polarDF=polarDF.reset_index()
polarDF=polarDF.rename(columns={'pos':'range_i'})
for i,r in polarDF.iterrows():
    polarDF.at[i,'RANGE'] = float(str(r['range_i'])[1:4].replace(',',''))
    polarDF.at[i,'Sentiment'] = 'positive'
del polarDF['range_i']
polarDF.head()

In [ ]:
polarDFneg=Eng_Reviews_DF[['neg']]
polarDFneg=polarDFneg.groupby(pd.cut(polarDFneg["neg"], np.arange(0, 1.1, 0.1))).count()
polarDFneg=polarDFneg.rename(columns={'neg':'count_of_Comments'})
polarDFneg=polarDFneg.reset_index()
polarDFneg=polarDFneg.rename(columns={'neg':'range_i'})
for i,r in polarDFneg.iterrows():
    polarDFneg.at[i,'RANGE'] = float(str(r['range_i'])[1:4].replace(',',''))
    polarDFneg.at[i,'Sentiment'] = 'negative'
del polarDFneg['range_i']
for i,r in polarDFneg.iterrows():
    polarDF=polarDF.append(pd.Series([r[0],r[1],r[2]],index=['count_of_Comments','RANGE','Sentiment']),ignore_index=True)

polarDFneg.head()

In [ ]:
polarDFneut=Eng_Reviews_DF[['neu']]
polarDFneut=polarDFneut.groupby(pd.cut(polarDFneut["neu"], np.arange(0, 1.0, 0.1))).count()
polarDFneut=polarDFneut.rename(columns={'neu':'count_of_Comments'})
polarDFneut=polarDFneut.reset_index()
polarDFneut=polarDFneut.rename(columns={'neu':'range_i'})
for i,r in polarDFneut.iterrows():
    polarDFneut.at[i,'RANGE'] = float(str(r['range_i'])[1:4].replace(',',''))
    polarDFneut.at[i,'Sentiment'] = 'neutral'
del polarDFneut['range_i']

for i,r in polarDFneut.iterrows():
    polarDF=polarDF.append(pd.Series([r[0],r[1],r[2]],index=['count_of_Comments','RANGE','Sentiment']),ignore_index=True)

polarDFneut.head()

In [ ]:
import seaborn as sns
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
sns.factorplot(data=polarDF, x="RANGE", y="count_of_Comments",col="Sentiment")

In [ ]:
inputDF = pd.read_csv('listings.csv')
inputDF=inputDF[['number_of_reviews','price','review_scores_rating']]

# replacing NaN values with 0
inputDF.fillna(0, inplace=True)

#Extracting prices from the table
price = inputDF['price']
prices=[]

#clean the data to make it float
for p in price:
    p=float(p[1:].replace(',',''))
    prices.append(p)

#replace the price column with the new column
inputDF['price']=prices

price_review = inputDF[['number_of_reviews', 'price']].sort_values(by = 'price')

price_review.plot(x = 'price',
                  y = 'number_of_reviews',
                  style = 'o',
                  figsize =(12,8),
                  legend = False,
                  title = 'Reviews based on Price')

plt.xlabel("price")
plt.ylabel("Number of reviews")

In [ ]:
sns.heatmap(price_review.corr(), vmin = -1, vmax = 1, linewidths = 1,
           annot = True, fmt = ".2f", annot_kws = {"size": 18}, cmap = "RdBu")

### Sentiment Analysis (Lexicon based approach - Unsupervised)

In [ ]:
#drop missing comments
review = reviews_DF.dropna(subset =['comments'],how='any')
review.shape[0]

In [ ]:
# Our dataframe consists of reviews in different language as well.Hence removing the comments which are not in english
from langdetect import detect

def detect_lang(sente):
    sente=str(sente)
    try:
        return detect(sente)
    except:
        return "None"

for index,row in review.iterrows():
    lang=detect_lang(row['comments'])
    review.at[index,'language'] = lang


#taking rows whose language is English
EngReviewsDF=review[review.language=='en']

In [ ]:
#Remove 'This is an automated posting' comments which are not real comments
EngReviewsDF['automated_posting'] = EngReviewsDF['comments'].apply(lambda row: "This is an automated posting" in row)
auto_posting = EngReviewsDF[EngReviewsDF['automated_posting']==True]
print('Sample auto comments:', auto_posting['comments'][83])
print('Number of auto comments:', auto_posting.shape[0])

In [ ]:
EngReviewsDF = EngReviewsDF[EngReviewsDF['automated_posting']==False]
print('Number of reviews after removing auto posting:', EngReviewsDF.shape[0])

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
from wordcloud import WordCloud,STOPWORDS, ImageColorGenerator
from sklearn.feature_extraction import text

review=EngReviewsDF
analyzer = SentimentIntensityAnalyzer()
review['review subjectivity Textblob']= review['comments'].apply(lambda row:TextBlob(row).sentiment.subjectivity)
review['review polarity Textblob']= review['comments'].apply(lambda row:TextBlob(row).sentiment.polarity)
review['review polarity Vader']= review['comments'].apply(lambda row:analyzer.polarity_scores(row)['compound'])
review['Average Polarity']=(review['review polarity Vader'] + review['review polarity Textblob'])/2

review['review Sentiment Textblob'] = np.where(review['review polarity Textblob']>= 0.01, 1, \
                                       (np.where(review['review polarity Textblob']<= -0.01, -1, 0)))

review['review Sentiment Vader'] = np.where(review['review polarity Vader']>= 0.05, 1, \
                                       (np.where(review['review polarity Vader']<=-0.05, -1, 0)))

print(review['review Sentiment Textblob'].value_counts())
print(review['review Sentiment Vader'].value_counts())

In [ ]:
review['Sentiment (lexicon)'] = np.where(review['Average Polarity']>0, 'Positive',\
                                   (np.where(review['Average Polarity']<0, 'Negative','Neutral')))
print('Combined sentiment from both methods:\n',review['Sentiment (lexicon)'].value_counts())

In [ ]:
review['Average Polarity'].describe()

In [ ]:
print('Random comments with the highest positive sentiment: \n')
cl_positive = review.loc[review['Average Polarity']>0.9,['comments']].sample(6).values
for c in cl_positive:
 print("*"+c[0])

In [ ]:
print('Random comments with the highest neutral sentiment: \n')
cl_neutral = review.loc[review['Average Polarity']==0,['comments']].sample(5).values
for c in cl_neutral:
 print("*"+c[0])

In [ ]:
print('Random comments with the highest negative sentiment: \n')
cl_negative = review.loc[review['Average Polarity']<-0.5,['comments']].sample(1).values
for c in cl_negative:
 print("*"+c[0])

### Top-grams analysis

In [ ]:
engStopword = set(text.ENGLISH_STOP_WORDS)
review['tidy_cm'] = review['comments'].apply(lambda x: ' '.join([w for w in str(x).split() if len(w)>1]))
review['tidy_cm'] = review['tidy_cm'].str.split().apply(lambda x: ' '.join(k for k in x if k.lower() not in engStopword))
review['tidy_cm'] = review['tidy_cm'].str.replace("n't", " not")
review['tidy_cm'] = review['tidy_cm'].str.replace("'s", " ")
review['tidy_cm'] = review['tidy_cm'].str.replace(r"[^a-zA-Z']", " ")

In [ ]:

def get_top_n_gram(corpus,gram, n):
    """Return n word for unigram, bigram, trigram, etc. from corpus

    Args:
        corpus (str): text to obtain the grams from
        gram (int): number of word in the phrase to extract. For example, 1: unigram, 2:bigram, 3:trigram
        n (int): number of phrases to be extracted from text. For example, 30 top unigrams, 20 top bigrams, etc.

    Returns:
        dataframe: a dataframe of top n words and their count in the text
        """

    vec = text.CountVectorizer(ngram_range=(gram, gram),stop_words = engStopword )
    bag_of_words = vec.fit_transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

In [ ]:
uni_grams = pd.DataFrame(get_top_n_gram(review['tidy_cm'],1,40), columns=['Words','count'])
bi_grams = pd.DataFrame(get_top_n_gram(review['tidy_cm'],2,30), columns=['Words','count'])
tri_grams = pd.DataFrame(get_top_n_gram(review['tidy_cm'],3,30), columns=['Words','count'])

In [ ]:
# Step 3: Visualize top grams
def plot_gram(data):
    """Visualize the top grams dataframe with seaborn barplot

    Args:
        data: dataframe of top words and their count returned from get_top_n_gram function

    Returns:
        barplot: barplot of top words and their count
    """

    data.sort_values(by=['count'], ascending = False)
    sns.set(rc={'figure.figsize':(12,7)})
    ax = sns.barplot(x='Words', y='count', data = data, palette = 'Blues_d');
    ax.set_xticklabels(labels = data['Words'], rotation=90);
    ax.set_title('Top grams from Reviews');

In [ ]:
plot_gram(uni_grams)

In [ ]:
plot_gram(bi_grams)

In [ ]:
plot_gram(tri_grams)

## Content based recommendation system

In [ ]:

from IPython.display import Image, HTML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
# Import dataset
listings_rec = pd.read_csv('listings.csv', usecols = ['id', 'name', 'description'])
listings_rec.head(10)

In [ ]:
listings_rec['name'] = listings_rec['name'].astype('str')
listings_rec['description'] = listings_rec['description'].astype('str')

In [ ]:
name_corpus = ' '.join(listings_rec['name'])
description_corpus = ' '.join(listings_rec['description'])

In [ ]:
name_wordcloud = WordCloud(stopwords = STOPWORDS, background_color = 'white', height = 2000, width = 4000).generate(name_corpus)
plt.figure(figsize = (16,8))
plt.imshow(name_wordcloud)
plt.axis('off')
plt.show()

In [ ]:
description_wordcloud = WordCloud(stopwords = STOPWORDS, background_color = 'white', height = 2000, width = 4000).generate(description_corpus)
plt.figure(figsize = (16,8))
plt.imshow(description_wordcloud)
plt.axis('off')
plt.show()

In [ ]:
listings_rec['content'] = listings_rec[['name', 'description']].astype(str).apply(lambda x: ' // '.join(x), axis = 1)

In [ ]:
# Fillna
listings_rec['content'].fillna('Null', inplace = True)

In [ ]:
tf = TfidfVectorizer(analyzer = 'word', ngram_range = (1, 2), min_df = 1, stop_words = 'english')
tfidf_matrix = tf.fit_transform(listings_rec['content'])

In [ ]:
cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix)

In [ ]:
results = {}
for idx, row in listings.iterrows():
    similar_indices = cosine_similarities[idx].argsort()[:-100:-1]
    similar_items = [(cosine_similarities[idx][i], listings['id'][i]) for i in similar_indices]
    results[row['id']] = similar_items[1:]

In [ ]:
def item(id):
    name   = listings_rec.loc[listings_rec['id'] == id]['content'].tolist()[0].split(' // ')[0]
    desc   = ' \nDescription: ' + listings_rec.loc[listings_rec['id'] == id]['content'].tolist()[0].split(' // ')[1][0:165] + '...'
    prediction = name  + desc
    return prediction

def recommend(item_id, num):
    print('Recommending ' + str(num) + ' products similar to ' + item(item_id))
    print('---')
    recs = results[item_id][:num]
    for rec in recs:
        print('\nRecommended: ' + item(rec[1]) + '\n(score:' + str(rec[0]) + ')')

In [ ]:
recommend(item_id = 3308979, num = 5)